In [0]:
# ========================================
# EXPLORATORY NOTEBOOK
# DATASET: gold_dim_sales_channel
# ========================================

# ========================================
# 0. CONFIGURATION
# ========================================

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "silver"
TARGET_SCHEMA = "gold"

DOMAIN = "dimensions"
DATASET = "gold_dim_sales_channel"

STORAGE_ACCOUNT = "stptfrozenfoodsdevwe01"
SILVER_CONTAINER = "silver"
GOLD_CONTAINER = "gold"

SALES_CHANNELS_DATASET = "reference_sales_channels"

SALES_CHANNELS_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{SALES_CHANNELS_DATASET}"

CANDIDATE_TARGET_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.dim_sales_channel"

SILVER_BASE_PATH = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
GOLD_BASE_PATH = f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

SALES_CHANNELS_SOURCE_PATH = f"{SILVER_BASE_PATH}reference/{SALES_CHANNELS_DATASET}/"
CANDIDATE_TARGET_PATH = f"{GOLD_BASE_PATH}dimensions/dim_sales_channel/"

In [0]:
# ========================================
# 1. CONTEXT SETUP
# ========================================

print("=" * 80)
print("STARTING GOLD EXPLORATORY: gold_dim_sales_channel")
print("=" * 80)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SOURCE_SCHEMA}")

print("[INFO] Context setup completed successfully.")

STARTING GOLD EXPLORATORY: gold_dim_sales_channel
[INFO] Context setup completed successfully.


In [0]:
# ========================================
# 2. CONFIGURATION SUMMARY
# ========================================

print("=" * 80)
print("GOLD EXPLORATORY NOTEBOOK CONFIGURATION")
print("=" * 80)
print(f"Catalog:                         {CATALOG}")
print(f"Source schema:                   {SOURCE_SCHEMA}")
print(f"Target schema:                   {TARGET_SCHEMA}")
print(f"Domain:                          {DOMAIN}")
print(f"Dataset:                         {DATASET}")
print(f"Sales channels table:            {SALES_CHANNELS_TABLE}")
print(f"Candidate target table:          {CANDIDATE_TARGET_TABLE}")
print(f"Sales channels source path:      {SALES_CHANNELS_SOURCE_PATH}")
print(f"Candidate target path:           {CANDIDATE_TARGET_PATH}")
print("=" * 80)

GOLD EXPLORATORY NOTEBOOK CONFIGURATION
Catalog:                         ptfrozenfoods_dev
Source schema:                   silver
Target schema:                   gold
Domain:                          dimensions
Dataset:                         gold_dim_sales_channel
Sales channels table:            ptfrozenfoods_dev.silver.reference_sales_channels
Candidate target table:          ptfrozenfoods_dev.gold.dim_sales_channel
Sales channels source path:      abfss://silver@stptfrozenfoodsdevwe01.dfs.core.windows.net/reference/reference_sales_channels/
Candidate target path:           abfss://gold@stptfrozenfoodsdevwe01.dfs.core.windows.net/dimensions/dim_sales_channel/


In [0]:
# ========================================
# 3. PRE-CHECKS
# ========================================

print("[INFO] Checking source table availability...")
spark.sql(f"DESCRIBE TABLE {SALES_CHANNELS_TABLE}")

print("[INFO] Checking source path access...")
dbutils.fs.ls(SALES_CHANNELS_SOURCE_PATH)

print("[INFO] Checking target container access...")
dbutils.fs.ls(f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/")

print("[INFO] Pre-checks completed successfully.")

[INFO] Checking source table availability...
[INFO] Checking source path access...
[INFO] Checking target container access...
[INFO] Pre-checks completed successfully.


In [0]:
# ========================================
# 4. READ SOURCE DATA
# ========================================

print("[INFO] Loading sales channels from the Silver layer...")

df_sales_channels = spark.table(SALES_CHANNELS_TABLE)

print("[INFO] Source tables loaded successfully.")

print("=" * 80)
print("SOURCE DATA ROW COUNTS")
print("=" * 80)
print(f"Sales Channels: {df_sales_channels.count():,}")
print("=" * 80)

# ========================================
# DATASET PREVIEW — INITIAL EXPLORATION
# ========================================

print("=" * 80)
print("DATASET PREVIEW — INITIAL EXPLORATION")
print("=" * 80)

print("\n[INFO] Preview: SALES CHANNELS (df_sales_channels)")
print("-" * 80)
display(df_sales_channels.limit(5))

print("\n[INFO] Printing schema...")
df_sales_channels.printSchema()

print("\n" + "=" * 80)
print("[INFO] Dataset preview completed successfully.")
print("=" * 80)

[INFO] Loading sales channels from the Silver layer...
[INFO] Source tables loaded successfully.
SOURCE DATA ROW COUNTS
Sales Channels: 4
DATASET PREVIEW — INITIAL EXPLORATION

[INFO] Preview: SALES CHANNELS (df_sales_channels)
--------------------------------------------------------------------------------


canal_id,nome_canal,descricao,load_date,ingestion_timestamp,source_file
CH02,Vendas Externas,Equipa comercial visitando clientes,2026-03-19,2026-04-02T16:53:14.226Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/reference/reference_sales_channels/load_date=2026-03-19/reference_sales_channels_20260319T151315Z.csv
CH04,Marketplace,Plataformas externas de venda,2026-03-19,2026-04-02T16:53:14.226Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/reference/reference_sales_channels/load_date=2026-03-19/reference_sales_channels_20260319T151315Z.csv
CH03,Telefone,Pedidos feitos por telefone,2026-03-19,2026-04-02T16:53:14.226Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/reference/reference_sales_channels/load_date=2026-03-19/reference_sales_channels_20260319T151315Z.csv
CH01,E-commerce,Pedidos realizados pelo site B2B/B2C,2026-03-19,2026-04-02T16:53:14.226Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/reference/reference_sales_channels/load_date=2026-03-19/reference_sales_channels_20260319T151315Z.csv



[INFO] Printing schema...
root
 |-- canal_id: string (nullable = true)
 |-- nome_canal: string (nullable = true)
 |-- descricao: string (nullable = true)
 |-- load_date: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)


[INFO] Dataset preview completed successfully.


In [0]:
# ========================================
# 5. SOURCE VALIDATION
# ========================================

from pyspark.sql import functions as F

required_columns = [
    "canal_id",
    "nome_canal"
]

missing_columns = [c for c in required_columns if c not in df_sales_channels.columns]

print(f"[INFO] Missing required columns: {missing_columns}")

if missing_columns:
    raise Exception(f"Missing required columns: {missing_columns}")

print("[INFO] Source validation completed successfully.")

[INFO] Missing required columns: []
[INFO] Source validation completed successfully.


In [0]:
# ========================================
# 6. INITIAL DATA QUALITY CHECKS
# ========================================

display(
    df_sales_channels.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("canal_id").alias("distinct_canal_id"),
        F.sum(F.when(F.col("canal_id").isNull(), 1).otherwise(0)).alias("null_canal_id")
    )
)

total_rows,distinct_canal_id,null_canal_id
4,4,0


In [0]:
# ========================================
# 7. DUPLICATE CHECK — SALES CHANNELS
# ========================================

df_duplicates = (
    df_sales_channels
    .groupBy("canal_id")
    .count()
    .filter(F.col("count") > 1)
)

display(df_duplicates)

canal_id,count


In [0]:
# ========================================
# 8. BUILD CANDIDATE DIMENSION
# ========================================

df_dim_sales_channel_candidate = (
    df_sales_channels
    .select(
        F.col("canal_id"),
        F.col("nome_canal"),
        F.col("descricao")
    )
    .dropDuplicates(["canal_id"])
)

print(f"[INFO] Candidate dimension row count: {df_dim_sales_channel_candidate.count():,}")
display(df_dim_sales_channel_candidate)

[INFO] Candidate dimension row count: 4


canal_id,nome_canal,descricao
CH01,E-commerce,Pedidos realizados pelo site B2B/B2C
CH02,Vendas Externas,Equipa comercial visitando clientes
CH03,Telefone,Pedidos feitos por telefone
CH04,Marketplace,Plataformas externas de venda


In [0]:
# ========================================
# 9. CANDIDATE DIMENSION VALIDATION
# ========================================

display(
    df_dim_sales_channel_candidate.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("canal_id").alias("distinct_canal_id"),
        F.sum(F.when(F.col("canal_id").isNull(), 1).otherwise(0)).alias("null_canal_id")
    )
)

print("[INFO] Candidate dimension validation completed successfully.")

total_rows,distinct_canal_id,null_canal_id
4,4,0


[INFO] Candidate dimension validation completed successfully.


In [0]:
# ========================================
# 10. MISSING VALUES ANALYSIS — DIM SALES CHANNEL
# ========================================

from pyspark.sql import functions as F

def analyze_missing_values(df, dataset_name):
    print("=" * 80)
    print(f"MISSING VALUES ANALYSIS — {dataset_name.upper()}")
    print("=" * 80)

    total_rows = df.count()

    missing_values_df = df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])

    missing_values_transposed = (
        missing_values_df
        .select(
            F.array([
                F.struct(
                    F.lit(col_name).alias("column_name"),
                    F.col(col_name).alias("null_count")
                ) for col_name in missing_values_df.columns
            ]).alias("missing_data")
        )
        .select(F.explode("missing_data").alias("data"))
        .select(
            F.col("data.column_name"),
            F.col("data.null_count")
        )
        .withColumn(
            "null_percentage",
            F.round((F.col("null_count") / F.lit(total_rows)) * 100, 4)
        )
        .orderBy(F.col("null_percentage").desc())
    )

    display(missing_values_transposed)

    print(f"[INFO] Total rows analyzed: {total_rows:,}")
    print("[INFO] Missing values analysis completed successfully.")
    print("=" * 80)


analyze_missing_values(df_dim_sales_channel_candidate, "dim_sales_channel")

MISSING VALUES ANALYSIS — DIM_SALES_CHANNEL


column_name,null_count,null_percentage
canal_id,0,0.0
nome_canal,0,0.0
descricao,0,0.0


[INFO] Total rows analyzed: 4
[INFO] Missing values analysis completed successfully.
